<a href="https://colab.research.google.com/github/Cris-19-77-99/Navier_Stokes_3D/blob/main/Navier_Stokes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

In [2]:
n_puntos=31
n_tiempos=1000

Lx=1
Ly=Lx

dx=Lx/(n_puntos-1)
dy=Ly/(n_puntos-1)

x=np.arange(0,Lx+dx,dx)
y=np.arange(0,Ly+dy,dy)

Re=100
alpha=0.8
alpha_p=0.8
viento=-0.1
nu=abs(viento)*Lx/Re

error_req=1e-3
errorP_req=1e-4
iter_max=10
iterV_max=100
iterP_max=1000

CFL=0.2
dt_conveccion=CFL*min(dx,dy)/max(abs(viento),1e-12)
dt_difusion=CFL/(2*nu*(1/dx**2+1/dy**2))
dt=min(dt_conveccion,dt_difusion)


u_t=np.zeros((n_tiempos,n_puntos+1,n_puntos))
v_t=np.zeros((n_tiempos,n_puntos,n_puntos+1))
p_t=np.ones((n_tiempos,n_puntos+1,n_puntos+1))


u_final=np.zeros((n_puntos-1,n_puntos-1))
v_final=np.zeros((n_puntos-1,n_puntos-1))
p_final=np.ones((n_puntos-1,n_puntos-1))


u=np.zeros((n_puntos+1,n_puntos))
u_estrella=np.zeros((n_puntos+1,n_puntos))
d_e=np.zeros((n_puntos+1,n_puntos))

v=np.zeros((n_puntos,n_puntos+1))
v_estrella=np.zeros((n_puntos,n_puntos+1))
d_n=np.zeros((n_puntos,n_puntos+1))

p=np.ones((n_puntos+1,n_puntos+1))
pc=np.zeros((n_puntos+1,n_puntos+1))
b=np.zeros((n_puntos+1,n_puntos+1))

u_nuevo=np.zeros((n_puntos+1,n_puntos))
v_nuevo=np.zeros((n_puntos,n_puntos+1))
p_nuevo=np.ones((n_puntos+1,n_puntos+1))


# Pared superior con viento
u[0,1:n_puntos-1]=viento
u_estrella[0,1:n_puntos-1]=viento
u_nuevo[0,1:n_puntos-1]=viento

u_t[0,:,:]=u
v_t[0,:,:]=v
p_t[0,:,:]=p

volumen=dx*dy
area_x=dy
area_y=dx


for r in range(1,n_tiempos):
  u_anterior=u_t[r-1,:,:].copy()
  v_anterior=v_t[r-1,:,:].copy()

  u=u_anterior.copy()
  v=v_anterior.copy()
  p=p_t[r-1,:,:].copy()

  error=1
  iteraciones=0

  while(error>error_req and iteraciones<iter_max):
    errorV=1
    iteracionesV=0

    while(errorV>error_req and iteracionesV<iterV_max):
      u_estrella=u.copy()
      v_estrella=v.copy()


      # Ecuacion para u
      u_E=0.5*(u[1:n_puntos,1:n_puntos-1]+u[1:n_puntos,2:n_puntos])
      u_W=0.5*(u[1:n_puntos,1:n_puntos-1]+u[1:n_puntos,0:n_puntos-2])
      v_N=0.5*(v[0:n_puntos-1,1:n_puntos-1]+v[0:n_puntos-1,2:n_puntos])
      v_S=0.5*(v[1:n_puntos,1:n_puntos-1]+v[1:n_puntos,2:n_puntos])

      a_E=-0.5*u_E*area_x+nu*area_x/dx
      a_W=0.5*u_W*area_x+nu*area_x/dx
      a_N=-0.5*v_N*area_y+nu*area_y/dy
      a_S=0.5*v_S*area_y+nu*area_y/dy

      a_e0=volumen/dt
      a_e=0.5*u_E*area_x-0.5*u_W*area_x+0.5*v_N*area_y-0.5*v_S*area_y+2*nu*(area_x/dx+area_y/dy)+a_e0

      A_e=-area_x
      d_e[1:n_puntos,1:n_puntos-1]=A_e/a_e

      u_estrella[1:n_puntos,1:n_puntos-1]=((a_E*u[1:n_puntos,2:n_puntos]+a_W*u[1:n_puntos,0:n_puntos-2]+a_N*u[0:n_puntos-1,1:n_puntos-1]+a_S*u[2:n_puntos+1,1:n_puntos-1]+a_e0*u_anterior[1:n_puntos,1:n_puntos-1])/a_e+d_e[1:n_puntos,1:n_puntos-1]*(p[1:n_puntos,2:n_puntos]-p[1:n_puntos,1:n_puntos-1]))

      del u_E,u_W,v_N,v_S


      # Ecuacion para v
      u_E=0.5*(u[1:n_puntos-1,1:n_puntos]+u[2:n_puntos,1:n_puntos])
      u_W=0.5*(u[1:n_puntos-1,0:n_puntos-1]+u[2:n_puntos,0:n_puntos-1])
      v_N=0.5*(v[0:n_puntos-2,1:n_puntos]+v[1:n_puntos-1,1:n_puntos])
      v_S=0.5*(v[1:n_puntos-1,1:n_puntos]+v[2:n_puntos,1:n_puntos])

      a_E=-0.5*u_E*area_x+nu*area_x/dx
      a_W=0.5*u_W*area_x+nu*area_x/dx
      a_N=-0.5*v_N*area_y+nu*area_y/dy
      a_S=0.5*v_S*area_y+nu*area_y/dy

      a_n0=volumen/dt
      a_n=0.5*u_E*area_x-0.5*u_W*area_x+0.5*v_N*area_y-0.5*v_S*area_y+2*nu*(area_x/dx+area_y/dy)+a_n0

      A_n=-area_y
      d_n[1:n_puntos-1,1:n_puntos]=A_n/a_n

      v_estrella[1:n_puntos-1,1:n_puntos]=((a_E*v[1:n_puntos-1,2:n_puntos+1]+a_W*v[1:n_puntos-1,0:n_puntos-1]+a_N*v[0:n_puntos-2,1:n_puntos]+a_S*v[2:n_puntos,1:n_puntos]+a_n0*v_anterior[1:n_puntos-1,1:n_puntos])/a_n+d_n[1:n_puntos-1,1:n_puntos]*(p[1:n_puntos-1,1:n_puntos]-p[2:n_puntos,1:n_puntos]))

      del u_E,u_W,v_N,v_S


      # Condiciones de borde
      u_estrella[0,:]=0
      u_estrella[n_puntos,:]=0
      u_estrella[:,0]=0
      u_estrella[:,n_puntos-1]=0
      u_estrella[0,1:n_puntos-1]=viento

      v_estrella[0,:]=0
      v_estrella[n_puntos-1,:]=0
      v_estrella[:,0]=0
      v_estrella[:,n_puntos]=0

      errorV=max(np.max(np.abs(u_estrella-u)),np.max(np.abs(v_estrella-v)))

      u=u_estrella.copy()
      v=v_estrella.copy()
      iteracionesV+=1


    # Correccion de la presion
    pc=np.zeros((n_puntos+1,n_puntos+1))
    pc_aux=np.zeros((n_puntos+1,n_puntos+1))

    a_E=-d_e[1:n_puntos,1:n_puntos]*area_x
    a_W=-d_e[1:n_puntos,0:n_puntos-1]*area_x
    a_N=-d_n[0:n_puntos-1,1:n_puntos]*area_y
    a_S=-d_n[1:n_puntos,1:n_puntos]*area_y
    a_P=a_E+a_W+a_N+a_S

    b[:,:]=0
    b[1:n_puntos,1:n_puntos]=-(u_estrella[1:n_puntos,1:n_puntos]-u_estrella[1:n_puntos,0:n_puntos-1])*area_x+(v_estrella[1:n_puntos,1:n_puntos]-v_estrella[0:n_puntos-1,1:n_puntos])*area_y

    errorP=1
    iteracionesP=0

    while(errorP>errorP_req and iteracionesP<iterP_max):
      pc_aux[1:n_puntos,1:n_puntos]=(a_E*pc[1:n_puntos,2:n_puntos+1]+a_W*pc[1:n_puntos,0:n_puntos-1]+a_N*pc[0:n_puntos-1,1:n_puntos]+a_S*pc[2:n_puntos+1,1:n_puntos]+b[1:n_puntos,1:n_puntos])/a_P

      pc_aux[1,1]=0

      errorP=np.max(np.abs(pc_aux-pc))
      pc=pc_aux.copy()
      iteracionesP+=1


    p_nuevo=p+alpha_p*pc

    p_nuevo[0,:]=p_nuevo[1,:]
    p_nuevo[n_puntos,:]=p_nuevo[n_puntos-1,:]
    p_nuevo[:,0]=p_nuevo[:,1]
    p_nuevo[:,n_puntos]=p_nuevo[:,n_puntos-1]

    p_nuevo=p_nuevo-(p_nuevo[1,1]-1)


    u_nuevo=u_estrella.copy()
    v_nuevo=v_estrella.copy()

    u_nuevo[1:n_puntos,1:n_puntos-1]=u_estrella[1:n_puntos,1:n_puntos-1]+alpha*d_e[1:n_puntos,1:n_puntos-1]*(pc[1:n_puntos,2:n_puntos]-pc[1:n_puntos,1:n_puntos-1])

    v_nuevo[1:n_puntos-1,1:n_puntos]=v_estrella[1:n_puntos-1,1:n_puntos]+alpha*d_n[1:n_puntos-1,1:n_puntos]*(pc[1:n_puntos-1,1:n_puntos]-pc[2:n_puntos,1:n_puntos])


    # Condiciones de borde pos corrección
    u_nuevo[0,:]=0
    u_nuevo[n_puntos,:]=0
    u_nuevo[:,0]=0
    u_nuevo[:,n_puntos-1]=0
    u_nuevo[0,1:n_puntos-1]=viento
    v_nuevo[0,:]=0
    v_nuevo[n_puntos-1,:]=0
    v_nuevo[:,0]=0
    v_nuevo[:,n_puntos]=0


    b[:,:]=0
    b[1:n_puntos,1:n_puntos]=-(u_nuevo[1:n_puntos,1:n_puntos]-u_nuevo[1:n_puntos,0:n_puntos-1])*area_x+(v_nuevo[1:n_puntos,1:n_puntos]-v_nuevo[0:n_puntos-1,1:n_puntos])*area_y

    error=np.max(np.abs(b[1:n_puntos,1:n_puntos]))

    u=u_nuevo.copy()
    v=v_nuevo.copy()
    p=p_nuevo.copy()
    iteraciones+=1


  if(r%100==0 or r==n_tiempos-1):
    print("paso ="+str(r)+", error continuidad ="+str(error)+", iteraciones ="+str(iteraciones))

  u_t[r,:,:]=u
  v_t[r,:,:]=v
  p_t[r,:,:]=p


x_centro=0.5*(x[0:n_puntos-1]+x[1:n_puntos])
y_centro=0.5*(y[0:n_puntos-1]+y[1:n_puntos])

u_final=0.5*(u[1:n_puntos,0:n_puntos-1]+u[1:n_puntos,1:n_puntos])
v_final=0.5*(v[0:n_puntos-1,1:n_puntos]+v[1:n_puntos,1:n_puntos])
p_final=p[1:n_puntos,1:n_puntos].copy()

print("dt ="+str(dt))
print("tiempo final ="+str((n_tiempos-1)*dt))

paso =100, error continuidad =0.00047627329369483965, iteraciones =1
paso =200, error continuidad =2.5042715115227348e-05, iteraciones =1
paso =300, error continuidad =1.0364100111664497e-05, iteraciones =1
paso =400, error continuidad =2.543597900038557e-05, iteraciones =1
paso =500, error continuidad =3.688603155256347e-05, iteraciones =1
paso =600, error continuidad =3.055703881388378e-05, iteraciones =1
paso =700, error continuidad =2.183580353959144e-05, iteraciones =1
paso =800, error continuidad =3.0618820111371734e-05, iteraciones =1
paso =900, error continuidad =3.526996245070932e-05, iteraciones =1
paso =999, error continuidad =3.224901245679636e-05, iteraciones =1
dt =0.05555555555555556
tiempo final =55.50000000000001


In [4]:
u_centro = 0.5 * (u_t[:, 0:n_puntos, :] + u_t[:, 1:n_puntos + 1, :])
v_centro = 0.5 * (v_t[:, :, 0:n_puntos] + v_t[:, :, 1:n_puntos + 1])

u_centro = u_centro[:, ::-1, :]
v_centro = v_centro[:, ::-1, :]


velocidad = np.sqrt(u_centro**2 + v_centro**2)
frames = np.linspace(0, n_tiempos - 1, min(100, n_tiempos), dtype=int)


X, Y = np.meshgrid(x, y)

# Recurrencia flechas
salto = 2
fig, ax = plt.subplots(figsize=(7, 6))
img = ax.imshow(velocidad[frames[0]],extent=[0, Lx, 0, Ly],origin="lower",cmap="turbo",vmin=0,vmax=np.max(velocidad[frames]),interpolation="bilinear")
flechas = ax.quiver(X[::salto, ::salto],Y[::salto, ::salto],u_centro[frames[0], ::salto, ::salto],v_centro[frames[0], ::salto, ::salto],color="white",pivot="mid")


barra = fig.colorbar(img, ax=ax)
barra.set_label("Módulo de la velocidad")
titulo = ax.set_title("Pared superior viento hacia la izquierda")


ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_xlim(0, Lx)
ax.set_ylim(0, Ly)
ax.set_aspect("equal")
ax.plot([0, Lx],[Ly, Ly])


def actualizar(frame):
    img.set_data(velocidad[frame])
    flechas.set_UVC(u_centro[frame, ::salto, ::salto],v_centro[frame, ::salto, ::salto])
    return img, flechas


animacion = FuncAnimation(fig,actualizar,frames=frames,interval=80,blit=False)
animacion.save("navier_stokes_2D.gif",writer=PillowWriter(fps=12))
plt.close(fig)

In [5]:
n_puntos=21
n_tiempos=100

Lx=1
Ly=Lx
Lz=Lx

dx=Lx/(n_puntos-1)
dy=Ly/(n_puntos-1)
dz=Lz/(n_puntos-1)

x=np.arange(0,Lx+dx,dx)
y=np.arange(0,Ly+dy,dy)
z=np.arange(0,Lz+dz,dz)

Re=100
alpha=0.8
alpha_p=0.8
viento=2
nu=abs(viento)*Lx/Re

error_req=1e-3
errorP_req=1e-4
iter_max=10
iterV_max=100
iterP_max=1000

CFL=0.2
dt_conveccion=CFL*min(dx,dy,dz)/max(abs(viento),1e-12)
dt_difusion=CFL/(2*nu*(1/dx**2+1/dy**2+1/dz**2))
dt=min(dt_conveccion,dt_difusion)


u_t=np.zeros((n_tiempos,n_puntos+1,n_puntos,n_puntos+1))
v_t=np.zeros((n_tiempos,n_puntos,n_puntos+1,n_puntos+1))
w_t=np.zeros((n_tiempos,n_puntos+1,n_puntos+1,n_puntos))
p_t=np.ones((n_tiempos,n_puntos+1,n_puntos+1,n_puntos+1))


u_final=np.zeros((n_puntos-1,n_puntos-1,n_puntos-1))
v_final=np.zeros((n_puntos-1,n_puntos-1,n_puntos-1))
w_final=np.zeros((n_puntos-1,n_puntos-1,n_puntos-1))
p_final=np.ones((n_puntos-1,n_puntos-1,n_puntos-1))


u=np.zeros((n_puntos+1,n_puntos,n_puntos+1))
u_estrella=np.zeros((n_puntos+1,n_puntos,n_puntos+1))
d_e=np.zeros((n_puntos+1,n_puntos,n_puntos+1))

v=np.zeros((n_puntos,n_puntos+1,n_puntos+1))
v_estrella=np.zeros((n_puntos,n_puntos+1,n_puntos+1))
d_n=np.zeros((n_puntos,n_puntos+1,n_puntos+1))

w=np.zeros((n_puntos+1,n_puntos+1,n_puntos))
w_estrella=np.zeros((n_puntos+1,n_puntos+1,n_puntos))
d_t=np.zeros((n_puntos+1,n_puntos+1,n_puntos))

p=np.ones((n_puntos+1,n_puntos+1,n_puntos+1))
pc=np.zeros((n_puntos+1,n_puntos+1,n_puntos+1))
b=np.zeros((n_puntos+1,n_puntos+1,n_puntos+1))

u_nuevo=np.zeros((n_puntos+1,n_puntos,n_puntos+1))
v_nuevo=np.zeros((n_puntos,n_puntos+1,n_puntos+1))
w_nuevo=np.zeros((n_puntos+1,n_puntos+1,n_puntos))
p_nuevo=np.ones((n_puntos+1,n_puntos+1,n_puntos+1))


# Viento en la tapa z=Lz
u[1:n_puntos,1:n_puntos-1,-1]=viento
u_estrella[1:n_puntos,1:n_puntos-1,-1]=viento
u_nuevo[1:n_puntos,1:n_puntos-1,-1]=viento

u_t[0,:,:,:]=u
v_t[0,:,:,:]=v
w_t[0,:,:,:]=w
p_t[0,:,:,:]=p

volumen=dx*dy*dz
area_x=dy*dz
area_y=dx*dz
area_z=dx*dy


for r in range(1,n_tiempos):
  u_anterior=u_t[r-1,:,:,:].copy()
  v_anterior=v_t[r-1,:,:,:].copy()
  w_anterior=w_t[r-1,:,:,:].copy()

  u=u_anterior.copy()
  v=v_anterior.copy()
  w=w_anterior.copy()
  p=p_t[r-1,:,:,:].copy()

  error=1
  iteraciones=0

  while(error>error_req and iteraciones<iter_max):
    errorV=1
    iteracionesV=0

    while(errorV>error_req and iteracionesV<iterV_max):
      u_estrella=u.copy()
      v_estrella=v.copy()
      w_estrella=w.copy()


      # Ecuacion para u
      u_E=0.5*(u[1:n_puntos,1:n_puntos-1,1:n_puntos]+u[1:n_puntos,2:n_puntos,1:n_puntos])
      u_W=0.5*(u[1:n_puntos,1:n_puntos-1,1:n_puntos]+u[1:n_puntos,0:n_puntos-2,1:n_puntos])
      v_N=0.5*(v[0:n_puntos-1,1:n_puntos-1,1:n_puntos]+v[0:n_puntos-1,2:n_puntos,1:n_puntos])
      v_S=0.5*(v[1:n_puntos,1:n_puntos-1,1:n_puntos]+v[1:n_puntos,2:n_puntos,1:n_puntos])
      w_T=0.5*(w[1:n_puntos,1:n_puntos-1,1:n_puntos]+w[1:n_puntos,2:n_puntos,1:n_puntos])
      w_B=0.5*(w[1:n_puntos,1:n_puntos-1,0:n_puntos-1]+w[1:n_puntos,2:n_puntos,0:n_puntos-1])

      a_E=-0.5*u_E*area_x+nu*area_x/dx
      a_W=0.5*u_W*area_x+nu*area_x/dx
      a_N=-0.5*v_N*area_y+nu*area_y/dy
      a_S=0.5*v_S*area_y+nu*area_y/dy
      a_T=-0.5*w_T*area_z+nu*area_z/dz
      a_B=0.5*w_B*area_z+nu*area_z/dz

      a_e0=volumen/dt
      a_e=0.5*u_E*area_x-0.5*u_W*area_x+0.5*v_N*area_y-0.5*v_S*area_y+0.5*w_T*area_z-0.5*w_B*area_z+2*nu*(area_x/dx+area_y/dy+area_z/dz)+a_e0

      A_e=-area_x
      d_e[1:n_puntos,1:n_puntos-1,1:n_puntos]=A_e/a_e

      u_estrella[1:n_puntos,1:n_puntos-1,1:n_puntos]=((a_E*u[1:n_puntos,2:n_puntos,1:n_puntos]+a_W*u[1:n_puntos,0:n_puntos-2,1:n_puntos]+a_N*u[0:n_puntos-1,1:n_puntos-1,1:n_puntos]+a_S*u[2:n_puntos+1,1:n_puntos-1,1:n_puntos]+a_T*u[1:n_puntos,1:n_puntos-1,2:n_puntos+1]+a_B*u[1:n_puntos,1:n_puntos-1,0:n_puntos-1]+a_e0*u_anterior[1:n_puntos,1:n_puntos-1,1:n_puntos])/a_e+d_e[1:n_puntos,1:n_puntos-1,1:n_puntos]*(p[1:n_puntos,2:n_puntos,1:n_puntos]-p[1:n_puntos,1:n_puntos-1,1:n_puntos]))

      del u_E,u_W,v_N,v_S,w_T,w_B


      # Ecuacion para v
      u_E=0.5*(u[1:n_puntos-1,1:n_puntos,1:n_puntos]+u[2:n_puntos,1:n_puntos,1:n_puntos])
      u_W=0.5*(u[1:n_puntos-1,0:n_puntos-1,1:n_puntos]+u[2:n_puntos,0:n_puntos-1,1:n_puntos])
      v_N=0.5*(v[0:n_puntos-2,1:n_puntos,1:n_puntos]+v[1:n_puntos-1,1:n_puntos,1:n_puntos])
      v_S=0.5*(v[1:n_puntos-1,1:n_puntos,1:n_puntos]+v[2:n_puntos,1:n_puntos,1:n_puntos])
      w_T=0.5*(w[1:n_puntos-1,1:n_puntos,1:n_puntos]+w[2:n_puntos,1:n_puntos,1:n_puntos])
      w_B=0.5*(w[1:n_puntos-1,1:n_puntos,0:n_puntos-1]+w[2:n_puntos,1:n_puntos,0:n_puntos-1])

      a_E=-0.5*u_E*area_x+nu*area_x/dx
      a_W=0.5*u_W*area_x+nu*area_x/dx
      a_N=-0.5*v_N*area_y+nu*area_y/dy
      a_S=0.5*v_S*area_y+nu*area_y/dy
      a_T=-0.5*w_T*area_z+nu*area_z/dz
      a_B=0.5*w_B*area_z+nu*area_z/dz

      a_n0=volumen/dt
      a_n=0.5*u_E*area_x-0.5*u_W*area_x+0.5*v_N*area_y-0.5*v_S*area_y+0.5*w_T*area_z-0.5*w_B*area_z+2*nu*(area_x/dx+area_y/dy+area_z/dz)+a_n0

      A_n=-area_y
      d_n[1:n_puntos-1,1:n_puntos,1:n_puntos]=A_n/a_n

      v_estrella[1:n_puntos-1,1:n_puntos,1:n_puntos]=((a_E*v[1:n_puntos-1,2:n_puntos+1,1:n_puntos]+a_W*v[1:n_puntos-1,0:n_puntos-1,1:n_puntos]+a_N*v[0:n_puntos-2,1:n_puntos,1:n_puntos]+a_S*v[2:n_puntos,1:n_puntos,1:n_puntos]+a_T*v[1:n_puntos-1,1:n_puntos,2:n_puntos+1]+a_B*v[1:n_puntos-1,1:n_puntos,0:n_puntos-1]+a_n0*v_anterior[1:n_puntos-1,1:n_puntos,1:n_puntos])/a_n+d_n[1:n_puntos-1,1:n_puntos,1:n_puntos]*(p[1:n_puntos-1,1:n_puntos,1:n_puntos]-p[2:n_puntos,1:n_puntos,1:n_puntos]))

      del u_E,u_W,v_N,v_S,w_T,w_B


      # Ecuacion para w
      u_E=0.5*(u[1:n_puntos,1:n_puntos,1:n_puntos-1]+u[1:n_puntos,1:n_puntos,2:n_puntos])
      u_W=0.5*(u[1:n_puntos,0:n_puntos-1,1:n_puntos-1]+u[1:n_puntos,0:n_puntos-1,2:n_puntos])
      v_N=0.5*(v[0:n_puntos-1,1:n_puntos,1:n_puntos-1]+v[0:n_puntos-1,1:n_puntos,2:n_puntos])
      v_S=0.5*(v[1:n_puntos,1:n_puntos,1:n_puntos-1]+v[1:n_puntos,1:n_puntos,2:n_puntos])
      w_T=0.5*(w[1:n_puntos,1:n_puntos,1:n_puntos-1]+w[1:n_puntos,1:n_puntos,2:n_puntos])
      w_B=0.5*(w[1:n_puntos,1:n_puntos,0:n_puntos-2]+w[1:n_puntos,1:n_puntos,1:n_puntos-1])

      a_E=-0.5*u_E*area_x+nu*area_x/dx
      a_W=0.5*u_W*area_x+nu*area_x/dx
      a_N=-0.5*v_N*area_y+nu*area_y/dy
      a_S=0.5*v_S*area_y+nu*area_y/dy
      a_T=-0.5*w_T*area_z+nu*area_z/dz
      a_B=0.5*w_B*area_z+nu*area_z/dz

      a_t0=volumen/dt
      a_t=0.5*u_E*area_x-0.5*u_W*area_x+0.5*v_N*area_y-0.5*v_S*area_y+0.5*w_T*area_z-0.5*w_B*area_z+2*nu*(area_x/dx+area_y/dy+area_z/dz)+a_t0

      A_t=-area_z
      d_t[1:n_puntos,1:n_puntos,1:n_puntos-1]=A_t/a_t

      w_estrella[1:n_puntos,1:n_puntos,1:n_puntos-1]=((a_E*w[1:n_puntos,2:n_puntos+1,1:n_puntos-1]+a_W*w[1:n_puntos,0:n_puntos-1,1:n_puntos-1]+a_N*w[0:n_puntos-1,1:n_puntos,1:n_puntos-1]+a_S*w[2:n_puntos+1,1:n_puntos,1:n_puntos-1]+a_T*w[1:n_puntos,1:n_puntos,2:n_puntos]+a_B*w[1:n_puntos,1:n_puntos,0:n_puntos-2]+a_t0*w_anterior[1:n_puntos,1:n_puntos,1:n_puntos-1])/a_t+d_t[1:n_puntos,1:n_puntos,1:n_puntos-1]*(p[1:n_puntos,1:n_puntos,2:n_puntos]-p[1:n_puntos,1:n_puntos,1:n_puntos-1]))

      del u_E,u_W,v_N,v_S,w_T,w_B


      # Condiciones de borde
      u_estrella[0,:,:]=0
      u_estrella[n_puntos,:,:]=0
      u_estrella[:,0,:]=0
      u_estrella[:,n_puntos-1,:]=0
      u_estrella[:,:,0]=0
      u_estrella[:,:,n_puntos]=0
      u_estrella[1:n_puntos,1:n_puntos-1,n_puntos]=viento

      v_estrella[0,:,:]=0
      v_estrella[n_puntos-1,:,:]=0
      v_estrella[:,0,:]=0
      v_estrella[:,n_puntos,:]=0
      v_estrella[:,:,0]=0
      v_estrella[:,:,n_puntos]=0

      w_estrella[0,:,:]=0
      w_estrella[n_puntos,:,:]=0
      w_estrella[:,0,:]=0
      w_estrella[:,n_puntos,:]=0
      w_estrella[:,:,0]=0
      w_estrella[:,:,n_puntos-1]=0

      errorV=max(np.max(np.abs(u_estrella-u)),np.max(np.abs(v_estrella-v)),np.max(np.abs(w_estrella-w)))

      u=u_estrella.copy()
      v=v_estrella.copy()
      w=w_estrella.copy()
      iteracionesV+=1


    # Correccion de presion
    pc=np.zeros((n_puntos+1,n_puntos+1,n_puntos+1))
    pc_aux=np.zeros((n_puntos+1,n_puntos+1,n_puntos+1))

    a_E=-d_e[1:n_puntos,1:n_puntos,1:n_puntos]*area_x
    a_W=-d_e[1:n_puntos,0:n_puntos-1,1:n_puntos]*area_x
    a_N=-d_n[0:n_puntos-1,1:n_puntos,1:n_puntos]*area_y
    a_S=-d_n[1:n_puntos,1:n_puntos,1:n_puntos]*area_y
    a_T=-d_t[1:n_puntos,1:n_puntos,1:n_puntos]*area_z
    a_B=-d_t[1:n_puntos,1:n_puntos,0:n_puntos-1]*area_z
    a_P=a_E+a_W+a_N+a_S+a_T+a_B

    b[:,:,:]=0
    b[1:n_puntos,1:n_puntos,1:n_puntos]=-(u_estrella[1:n_puntos,1:n_puntos,1:n_puntos]-u_estrella[1:n_puntos,0:n_puntos-1,1:n_puntos])*area_x+(v_estrella[1:n_puntos,1:n_puntos,1:n_puntos]-v_estrella[0:n_puntos-1,1:n_puntos,1:n_puntos])*area_y-(w_estrella[1:n_puntos,1:n_puntos,1:n_puntos]-w_estrella[1:n_puntos,1:n_puntos,0:n_puntos-1])*area_z

    errorP=1
    iteracionesP=0

    while(errorP>errorP_req and iteracionesP<iterP_max):
      pc_aux[1:n_puntos,1:n_puntos,1:n_puntos]=(a_E*pc[1:n_puntos,2:n_puntos+1,1:n_puntos]+a_W*pc[1:n_puntos,0:n_puntos-1,1:n_puntos]+a_N*pc[0:n_puntos-1,1:n_puntos,1:n_puntos]+a_S*pc[2:n_puntos+1,1:n_puntos,1:n_puntos]+a_T*pc[1:n_puntos,1:n_puntos,2:n_puntos+1]+a_B*pc[1:n_puntos,1:n_puntos,0:n_puntos-1]+b[1:n_puntos,1:n_puntos,1:n_puntos])/a_P

      # Presion de referencia
      pc_aux[1,1,1]=0

      errorP=np.max(np.abs(pc_aux-pc))
      pc=pc_aux.copy()
      iteracionesP+=1


    p_nuevo=p+alpha_p*pc

    p_nuevo[0,:,:]=p_nuevo[1,:,:]
    p_nuevo[n_puntos,:,:]=p_nuevo[n_puntos-1,:,:]
    p_nuevo[:,0,:]=p_nuevo[:,1,:]
    p_nuevo[:,n_puntos,:]=p_nuevo[:,n_puntos-1,:]
    p_nuevo[:,:,0]=p_nuevo[:,:,1]
    p_nuevo[:,:,n_puntos]=p_nuevo[:,:,n_puntos-1]

    p_nuevo=p_nuevo-(p_nuevo[1,1,1]-1)


    u_nuevo=u_estrella.copy()
    v_nuevo=v_estrella.copy()
    w_nuevo=w_estrella.copy()

    u_nuevo[1:n_puntos,1:n_puntos-1,1:n_puntos]=u_estrella[1:n_puntos,1:n_puntos-1,1:n_puntos]+alpha*d_e[1:n_puntos,1:n_puntos-1,1:n_puntos]*(pc[1:n_puntos,2:n_puntos,1:n_puntos]-pc[1:n_puntos,1:n_puntos-1,1:n_puntos])

    v_nuevo[1:n_puntos-1,1:n_puntos,1:n_puntos]=v_estrella[1:n_puntos-1,1:n_puntos,1:n_puntos]+alpha*d_n[1:n_puntos-1,1:n_puntos,1:n_puntos]*(pc[1:n_puntos-1,1:n_puntos,1:n_puntos]-pc[2:n_puntos,1:n_puntos,1:n_puntos])

    w_nuevo[1:n_puntos,1:n_puntos,1:n_puntos-1]=w_estrella[1:n_puntos,1:n_puntos,1:n_puntos-1]+alpha*d_t[1:n_puntos,1:n_puntos,1:n_puntos-1]*(pc[1:n_puntos,1:n_puntos,2:n_puntos]-pc[1:n_puntos,1:n_puntos,1:n_puntos-1])


    # Condiciones de borde despues de la correccion
    u_nuevo[0,:,:]=0
    u_nuevo[n_puntos,:,:]=0
    u_nuevo[:,0,:]=0
    u_nuevo[:,n_puntos-1,:]=0
    u_nuevo[:,:,0]=0
    u_nuevo[:,:,n_puntos]=0
    u_nuevo[1:n_puntos,1:n_puntos-1,n_puntos]=viento

    v_nuevo[0,:,:]=0
    v_nuevo[n_puntos-1,:,:]=0
    v_nuevo[:,0,:]=0
    v_nuevo[:,n_puntos,:]=0
    v_nuevo[:,:,0]=0
    v_nuevo[:,:,n_puntos]=0

    w_nuevo[0,:,:]=0
    w_nuevo[n_puntos,:,:]=0
    w_nuevo[:,0,:]=0
    w_nuevo[:,n_puntos,:]=0
    w_nuevo[:,:,0]=0
    w_nuevo[:,:,n_puntos-1]=0


    b[:,:,:]=0
    b[1:n_puntos,1:n_puntos,1:n_puntos]=-(u_nuevo[1:n_puntos,1:n_puntos,1:n_puntos]-u_nuevo[1:n_puntos,0:n_puntos-1,1:n_puntos])*area_x+(v_nuevo[1:n_puntos,1:n_puntos,1:n_puntos]-v_nuevo[0:n_puntos-1,1:n_puntos,1:n_puntos])*area_y-(w_nuevo[1:n_puntos,1:n_puntos,1:n_puntos]-w_nuevo[1:n_puntos,1:n_puntos,0:n_puntos-1])*area_z

    error=np.max(np.abs(b[1:n_puntos,1:n_puntos,1:n_puntos]))

    u=u_nuevo.copy()
    v=v_nuevo.copy()
    w=w_nuevo.copy()
    p=p_nuevo.copy()
    iteraciones+=1


  if(r%10==0 or r==n_tiempos-1):
    print("paso ="+str(r)+", error continuidad ="+str(error)+", iteraciones ="+str(iteraciones))

  u_t[r,:,:,:]=u
  v_t[r,:,:,:]=v
  w_t[r,:,:,:]=w
  p_t[r,:,:,:]=p


x_centro=0.5*(x[0:n_puntos-1]+x[1:n_puntos])
y_centro=0.5*(y[0:n_puntos-1]+y[1:n_puntos])
z_centro=0.5*(z[0:n_puntos-1]+z[1:n_puntos])

u_final=0.5*(u[1:n_puntos,0:n_puntos-1,1:n_puntos]+u[1:n_puntos,1:n_puntos,1:n_puntos])
v_final=0.5*(v[0:n_puntos-1,1:n_puntos,1:n_puntos]+v[1:n_puntos,1:n_puntos,1:n_puntos])
w_final=0.5*(w[1:n_puntos,1:n_puntos,0:n_puntos-1]+w[1:n_puntos,1:n_puntos,1:n_puntos])
p_final=p[1:n_puntos,1:n_puntos,1:n_puntos].copy()

print("dt ="+str(dt))
print("tiempo final ="+str((n_tiempos-1)*dt))

paso =10, error continuidad =0.00012530275072106836, iteraciones =1
paso =20, error continuidad =0.00011870870707512275, iteraciones =1
paso =30, error continuidad =7.777465119739046e-05, iteraciones =1
paso =40, error continuidad =3.0829151363255326e-06, iteraciones =1
paso =50, error continuidad =3.26194768981464e-05, iteraciones =1
paso =60, error continuidad =5.036272108160024e-06, iteraciones =1
paso =70, error continuidad =5.218753576258413e-06, iteraciones =1
paso =80, error continuidad =2.8994166760851874e-06, iteraciones =1
paso =90, error continuidad =5.83517637494528e-06, iteraciones =1
paso =99, error continuidad =3.4239864275638183e-06, iteraciones =1
dt =0.0041666666666666675
tiempo final =0.4125000000000001


In [6]:
u_centro = 0.5 * (u_t[:, 1:n_puntos, 0:n_puntos-1, 1:n_puntos] + u_t[:, 1:n_puntos, 1:n_puntos, 1:n_puntos])
v_centro = 0.5 * (v_t[:, 0:n_puntos-1, 1:n_puntos, 1:n_puntos] + v_t[:, 1:n_puntos, 1:n_puntos, 1:n_puntos])
w_centro = 0.5 * (w_t[:, 1:n_puntos, 1:n_puntos, 0:n_puntos-1] + w_t[:, 1:n_puntos, 1:n_puntos, 1:n_puntos])


# Ordenar los arreglos
u_centro = np.transpose(u_centro, (0, 2, 1, 3))
v_centro = np.transpose(v_centro, (0, 2, 1, 3))
w_centro = np.transpose(w_centro, (0, 2, 1, 3))


velocidad = np.sqrt(u_centro**2 + v_centro**2 + w_centro**2)
frames = np.linspace(0, n_tiempos-1, min(100, n_tiempos), dtype=int)


x_centro = 0.5 * (x[:-1] + x[1:])
y_centro = 0.5 * (y[:-1] + y[1:])
z_centro = 0.5 * (z[:-1] + z[1:])

X, Y, Z = np.meshgrid(x_centro, y_centro, z_centro, indexing="ij")


# Recurrencia flechas
salto = max(1, (n_puntos-1)//7)

Xq = X[::salto, ::salto, ::salto]
Yq = Y[::salto, ::salto, ::salto]
Zq = Z[::salto, ::salto, ::salto]


vmax = np.max(velocidad[frames])
if vmax == 0:
    vmax = 1
cmap = plt.cm.turbo
norma = Normalize(vmin=0, vmax=vmax)
mapa_color = ScalarMappable(norm=norma, cmap=cmap)
mapa_color.set_array([])


fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")

barra = fig.colorbar(mapa_color, ax=ax, shrink=0.7, pad=0.1)
barra.set_label("Módulo de la velocidad")


vertices = np.array([
    [0, 0, 0],
    [Lx, 0, 0],
    [Lx, Ly, 0],
    [0, Ly, 0],
    [0, 0, Lz],
    [Lx, 0, Lz],
    [Lx, Ly, Lz],
    [0, Ly, Lz]
])

aristas = [
    [0, 1], [1, 2], [2, 3], [3, 0],
    [4, 5], [5, 6], [6, 7], [7, 4],
    [0, 4], [1, 5], [2, 6], [3, 7]
]


def actualizar(frame):

    ax.cla()

    U = u_centro[frame, ::salto, ::salto, ::salto]
    V = v_centro[frame, ::salto, ::salto, ::salto]
    W = w_centro[frame, ::salto, ::salto, ::salto]

    modulo = velocidad[frame, ::salto, ::salto, ::salto]
    colores = cmap(norma(modulo.flatten()))
    colores = np.concatenate((colores, colores, colores), axis=0)

    ax.quiver(
        Xq,
        Yq,
        Zq,
        U / abs(viento),
        V / abs(viento),
        W / abs(viento),
        length=0.25,
        normalize=False,
        color=colores,
        linewidth=1,
        arrow_length_ratio=0.3
    )


    for arista in aristas:

        inicio = vertices[arista[0]]
        final = vertices[arista[1]]

        ax.plot(
            [inicio[0], final[0]],
            [inicio[1], final[1]],
            [inicio[2], final[2]],
            color="black"
        )


    ax.quiver(
        0.3 * Lx,
        0.5 * Ly,
        Lz,
        np.sign(viento),
        0,
        0,
        length=0.4 * Lx,
        normalize=True,
        color="red",
        linewidth=3
    )


    ax.set_title(
        "Velocidades en cubo 3D = "
        + str(round(frame * dt, 3))
    )

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

    ax.set_xlim(0, Lx)
    ax.set_ylim(0, Ly)
    ax.set_zlim(0, Lz)

    ax.set_box_aspect((Lx, Ly, Lz))
    ax.view_init(elev=25, azim=35)

    return ax
animacion = FuncAnimation(fig,actualizar,frames=frames,interval=80,blit=False)
animacion.save("navier_stokes_3D.gif",writer=PillowWriter(fps=12))

plt.close(fig)